<a href="https://colab.research.google.com/github/zeynepc22/ISYS2001-ZeynepCevik/blob/main/Module%2003/lab_ticket_budget_decisions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/kevin-blasiak-curtin/ISYS2001-Archive/blob/main/Module%2003%20-%20Making%20Computers%20Think/lab_ticket_budget_decisions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 Lab Ticket: Budget Decisions

This week your job is not to hand-code a program. It is to write **one complete, detailed prompt** that an AI can turn into a working budget tool in a single go.

In the worksheet you practised giving AI clear intent. Here that is the whole task. The skill being tested is whether you can describe exactly what you want, in enough detail, that one prompt produces the finished tool with no back-and-forth.

## What to submit

1. **Your final prompt**: one complete, self-contained prompt (paste it into the prompt cell below).
2. **The code it produced**: paste the AI-generated program into the code cell and run it, so you can show it works.

Your prompt can be the result of several rounds of refining. What you hand in is the single finished version: the one that would generate the whole tool if someone ran it cold.

## The brief

Design a tool that helps someone make a smart budget decision. What that means, and who it is for, is up to you. Some directions, though you are not limited to these:

- An expense classifier that sorts spending into categories and reacts to each.
- A budget checker that warns when an expense is too large a share of someone's budget.
- A savings goal tracker that gives different feedback depending on progress.
- A purchase advisor that weighs a price against how much money and income the person has.

Pick one or invent your own. The only hard requirement is that the finished program makes genuinely different decisions depending on the numbers it is given.

## How this works

You are going to write a prompt, hand it to an AI, and see what it builds. Your first attempt will not be right. Read what comes back and look for the gaps:

- Did it invent a rule you did not ask for? Your prompt was not precise enough.
- Did it miss a case? Say so explicitly next time.
- Did it ask you follow-up questions? A complete prompt should not need any.

Fold each fix back into the prompt and run it again. Keep going until one single prompt produces the whole working tool in one go. That final version is what you submit.

How detailed does it need to be? Detailed enough that a classmate could paste it cold and get the same working tool. Work out for yourself what that takes.

In [11]:
import re


# ============================================================
# CATEGORIES
# ============================================================

CATEGORIES = [
    "Food",
    "Shopping",
    "Groceries",
    "Entertainment",
    "Bills",
    "Rent/Mortgage",
    "Uncategorised"
]


# ============================================================
# PATTERNS USED FOR AUTOMATIC CATEGORISATION
# ============================================================
# The program checks the user's description against these
# patterns. More specific patterns are used where necessary.
# ============================================================

PATTERNS = {

    # --------------------------------------------------------
    # Rent / Mortgage
    # --------------------------------------------------------
    "Rent/Mortgage": [
        r"\brent\b",
        r"\brent payment\b",
        r"\brental payment\b",
        r"\bmortgage\b",
        r"\bmortgage payment\b",
        r"\bmortgage repayment\b",
        r"\bhome loan\b",
        r"\bhouse payment\b"
    ],

    # --------------------------------------------------------
    # Groceries
    # --------------------------------------------------------
    "Groceries": [
        r"\bcoles\b",
        r"\bwoolworths\b",
        r"\bwoolies\b",
        r"\baldi\b",
        r"\biga\b",
        r"\bsupermarket\b",
        r"\bsupermarkets\b",
        r"\bgroceries\b",
        r"\bgrocery shopping\b",
        r"\bgrocery\b",
        r"\bweekly shop\b",
        r"\bfood shopping\b"
    ],

    # --------------------------------------------------------
    # Entertainment
    # --------------------------------------------------------
    "Entertainment": [
        r"\bnetflix\b",
        r"\bspotify\b",
        r"\bdisney\s*\+?\b",
        r"\bprime video\b",
        r"\byoutube premium\b",
        r"\bcinema\b",
        r"\bmovie\b",
        r"\bmovies\b",
        r"\bmovie ticket\b",
        r"\bconcert\b",
        r"\bconcert ticket\b",
        r"\btheatre\b",
        r"\btheater\b",
        r"\btheme park\b",
        r"\bamusement park\b",
        r"\bgaming\b",
        r"\bvideo game\b",
        r"\bvideo games\b",
        r"\bgame purchase\b",
        r"\bsteam\b"
    ],

    # --------------------------------------------------------
    # Bills
    # --------------------------------------------------------
    "Bills": [
        r"\belectricity\b",
        r"\belectricity bill\b",
        r"\belectric bill\b",
        r"\bgas bill\b",
        r"\bgas payment\b",
        r"\bwater bill\b",
        r"\bwater payment\b",
        r"\bwifi\b",
        r"\bwi-fi\b",
        r"\binternet bill\b",
        r"\binternet payment\b",
        r"\bphone bill\b",
        r"\bmobile bill\b",
        r"\btelephone bill\b",
        r"\butility bill\b",
        r"\butilities\b"
    ],

    # --------------------------------------------------------
    # Shopping
    # --------------------------------------------------------
    "Shopping": [
        r"\bclothes\b",
        r"\bclothing\b",
        r"\bdress\b",
        r"\bjeans\b",
        r"\bshirt\b",
        r"\bshirts\b",
        r"\btop\b",
        r"\btops\b",
        r"\bjacket\b",
        r"\bcoat\b",
        r"\bshoes\b",
        r"\bshoe\b",
        r"\bsneakers\b",
        r"\btrainers\b",
        r"\bboots\b",
        r"\belectronics\b",
        r"\belectronic\b",
        r"\blaptop\b",
        r"\bcomputer\b",
        r"\btablet\b",
        r"\bheadphones\b",
        r"\bearphones\b",
        r"\bkeyboard\b",
        r"\bmouse\b",
        r"\bmonitor\b",
        r"\bcamera\b",
        r"\btelevision\b",
        r"\btv\b",
        r"\biphone\b",
        r"\bipad\b",
        r"\bsamsung\b",
        r"\bnike\b",
        r"\badidas\b",
        r"\buniqlo\b",
        r"\bzara\b",
        r"\bh&m\b",
        r"\bkmart\b",
        r"\btarget\b",
        r"\bmyer\b",
        r"\bdavid jones\b",
        r"\bshopping\b",
        r"\bpurchase\b",
        r"\bnew clothes\b",
        r"\bnew shoes\b",
        r"\bnew phone\b",
        r"\bnew laptop\b"
    ],

    # --------------------------------------------------------
    # Food
    # --------------------------------------------------------
    # These patterns focus on eating OUT rather than groceries.
    # --------------------------------------------------------
    "Food": [
        r"\brestaurant\b",
        r"\bdining out\b",
        r"\bdine out\b",
        r"\bdinner\b",
        r"\blunch\b",
        r"\bbreakfast\b",
        r"\bbrunch\b",
        r"\bcafe\b",
        r"\bcafé\b",
        r"\bcoffee shop\b",
        r"\bcoffee\b",
        r"\bpizza\b",
        r"\btakeaway\b",
        r"\btakeout\b",
        r"\bfast food\b",
        r"\bmcdonald'?s\b",
        r"\bkfc\b",
        r"\bhungry jacks\b",
        r"\bsubway\b",
        r"\bdomino'?s\b",
        r"\budon\b",
        r"\bsushi\b",
        r"\bthai restaurant\b",
        r"\bindian restaurant\b",
        r"\bchinese restaurant\b",
        r"\buber eats\b",
        r"\bdoordash\b",
        r"\bmenulog\b"
    ]
}


# ============================================================
# AUTOMATIC CATEGORISATION
# ============================================================

def categorise_expense(description):
    """
    Checks the user's description against the patterns
    and returns the most appropriate category.

    If no pattern matches, the expense is placed into
    Uncategorised.
    """

    description = description.lower().strip()

    # Check each category
    for category, patterns in PATTERNS.items():

        for pattern in patterns:

            if re.search(pattern, description):
                return category

    # Nothing matched
    return "Uncategorised"


# ============================================================
# GET A VALID AMOUNT
# ============================================================

def get_amount(prompt):
    """
    Keeps asking until the user enters a valid
    positive number.
    """

    while True:

        try:
            amount = float(input(prompt))

            if amount <= 0:
                print("Please enter an amount greater than $0.")

            else:
                return amount

        except ValueError:
            print("Invalid input. Please enter a number.")
            print("Example: 50 or 50.50")


# ============================================================
# ADD AN EXPENSE
# ============================================================

def add_expense(expenses):

    print("\n" + "=" * 65)
    print("ADD EXPENSE")
    print("=" * 65)

    # Get amount
    amount = get_amount("Enter expense amount: $")

    # Get description
    description = input(
        "What did you spend it on? "
    ).strip()

    # Don't allow an empty description
    while description == "":
        print("Description cannot be empty.")

        description = input(
            "What did you spend it on? "
        ).strip()

    # Automatically categorise the description
    category = categorise_expense(description)

    print("\nAutomatic categorisation:")
    print(f"Description : {description}")
    print(f"Category    : {category}")

    # Store expense
    expense = {
        "amount": amount,
        "description": description,
        "category": category
    }

    expenses.append(expense)

    print("\nExpense added successfully.")


# ============================================================
# SAMPLE DATA
# ============================================================

def load_sample_data():

    return [
        {
            "amount": 45.50,
            "description": "Dinner at Italian restaurant",
            "category": "Food"
        },
        {
            "amount": 125.75,
            "description": "Weekly groceries at Coles",
            "category": "Groceries"
        },
        {
            "amount": 89.99,
            "description": "New pair of Nike shoes",
            "category": "Shopping"
        },
        {
            "amount": 19.99,
            "description": "Netflix subscription",
            "category": "Entertainment"
        },
        {
            "amount": 145.00,
            "description": "Electricity bill",
            "category": "Bills"
        },
        {
            "amount": 600.00,
            "description": "August rent",
            "category": "Rent/Mortgage"
        },
        {
            "amount": 35.00,
            "description": "Cinema tickets",
            "category": "Entertainment"
        },
        {
            "amount": 65.00,
            "description": "New headphones",
            "category": "Shopping"
        },
        {
            "amount": 15.00,
            "description": "Coffee with friends",
            "category": "Food"
        },
        {
            "amount": 50.00,
            "description": "Uber to university",
            "category": "Uncategorised"
        }
    ]


# ============================================================
# DISPLAY ALL EXPENSES
# ============================================================

def display_expenses(expenses):

    print("\n" + "=" * 80)
    print("ALL EXPENSES")
    print("=" * 80)

    if len(expenses) == 0:
        print("No expenses have been recorded.")
        return

    print(
        f"{'No.':<5}"
        f"{'Description':<35}"
        f"{'Amount':>15}"
        f"{'Category':>20}"
    )

    print("-" * 80)

    for number, expense in enumerate(expenses, start=1):

        print(
            f"{number:<5}"
            f"{expense['description']:<35}"
            f"${expense['amount']:>13.2f}"
            f"{expense['category']:>20}"
        )


# ============================================================
# DISPLAY SUMMARY
# ============================================================

def display_summary(expenses, salary):

    # Create a total for every category
    totals = {}

    for category in CATEGORIES:
        totals[category] = 0

    # Add each expense to the appropriate category
    for expense in expenses:

        category = expense["category"]

        totals[category] += expense["amount"]

    # Calculate total spent
    total_spent = sum(totals.values())

    # Calculate remaining salary
    remaining = salary - total_spent

    # Sort categories from highest spending to lowest
    sorted_totals = sorted(
        totals.items(),
        key=lambda item: item[1],
        reverse=True
    )

    print("\n" + "=" * 80)
    print("MONTHLY BUDGET SUMMARY")
    print("=" * 80)

    print(f"Monthly Salary : ${salary:,.2f}")
    print(f"Total Spent    : ${total_spent:,.2f}")
    print(f"Remaining      : ${remaining:,.2f}")

    print("\n" + "-" * 80)

    print(
        f"{'Category':<25}"
        f"{'Total':>20}"
        f"{'% of Spending':>25}"
    )

    print("-" * 80)

    for category, total in sorted_totals:

        # Avoid division by zero
        if total_spent > 0:
            percentage = (total / total_spent) * 100
        else:
            percentage = 0

        print(
            f"{category:<25}"
            f"${total:>19,.2f}"
            f"{percentage:>23.2f}%"
        )

    print("-" * 80)


# ============================================================
# RECATEGORISE AN EXPENSE
# ============================================================

def recategorise_expense(expenses):

    if len(expenses) == 0:
        print("\nThere are no expenses to recategorise.")
        return

    # Display expenses so the user can choose one
    display_expenses(expenses)

    print("\nEnter 0 to cancel.")

    while True:

        try:

            choice = int(
                input(
                    "\nEnter the expense number "
                    "to recategorise: "
                )
            )

            # Cancel
            if choice == 0:
                print("Recategorisation cancelled.")
                return

            # Check whether the number is valid
            if choice < 1 or choice > len(expenses):

                print(
                    f"Please enter a number from "
                    f"1 to {len(expenses)}."
                )

                continue

            # Select expense
            expense = expenses[choice - 1]

            print("\nSelected expense:")
            print(f"Description : {expense['description']}")
            print(f"Amount      : ${expense['amount']:.2f}")
            print(f"Category    : {expense['category']}")

            # Show available categories
            print("\nAvailable categories:")

            for number, category in enumerate(
                CATEGORIES,
                start=1
            ):
                print(f"{number}. {category}")

            # Get new category
            while True:

                try:

                    category_choice = int(
                        input(
                            "\nChoose the new category: "
                        )
                    )

                    if (
                        category_choice < 1
                        or category_choice > len(CATEGORIES)
                    ):

                        print(
                            f"Please enter a number from "
                            f"1 to {len(CATEGORIES)}."
                        )

                        continue

                    # Change category
                    new_category = CATEGORIES[
                        category_choice - 1
                    ]

                    expense["category"] = new_category

                    print(
                        f"\nExpense successfully "
                        f"recategorised as "
                        f"'{new_category}'."
                    )

                    return

                except ValueError:

                    print(
                        "Invalid input. Please enter a number."
                    )

        except ValueError:

            print(
                "Invalid input. Please enter a number."
            )


# ============================================================
# MAIN PROGRAM
# ============================================================

def main():

    print("=" * 65)
    print("             MONTHLY BUDGET PROGRAM")
    print("=" * 65)

    # --------------------------------------------------------
    # Ask for salary
    # --------------------------------------------------------

    salary = get_amount(
        "\nEnter your salary for this month: $"
    )

    # List containing all expenses
    expenses = []

    # --------------------------------------------------------
    # Main menu
    # --------------------------------------------------------

    while True:

        print("\n" + "=" * 65)
        print("MENU")
        print("=" * 65)

        print("1. Add expense")
        print("2. Load sample data")
        print("3. Display summary")
        print("4. Recategorise expense")
        print("5. Display all expenses")
        print("6. Exit")

        choice = input(
            "\nChoose an option (1-6): "
        ).strip()

        # ----------------------------------------------------
        # 1. Add expense
        # ----------------------------------------------------

        if choice == "1":

            add_expense(expenses)

        # ----------------------------------------------------
        # 2. Load sample data
        # ----------------------------------------------------

        elif choice == "2":

            expenses = load_sample_data()

            print(
                "\nSample data loaded successfully."
            )

        # ----------------------------------------------------
        # 3. Display summary
        # ----------------------------------------------------

        elif choice == "3":

            display_summary(
                expenses,
                salary
            )

        # ----------------------------------------------------
        # 4. Recategorise
        # ----------------------------------------------------

        elif choice == "4":

            recategorise_expense(expenses)

        # ----------------------------------------------------
        # 5. Display expenses
        # ----------------------------------------------------

        elif choice == "5":

            display_expenses(expenses)

        # ----------------------------------------------------
        # 6. Exit
        # ----------------------------------------------------

        elif choice == "6":

            print(
                "\nThank you for using "
                "the Budget Program!"
            )

            break

        # ----------------------------------------------------
        # Invalid option
        # ----------------------------------------------------

        else:

            print(
                "\nInvalid option. "
                "Please choose a number from 1 to 6."
            )


# ============================================================
# RUN PROGRAM
# ============================================================

if __name__ == "__main__":
    main()


             MONTHLY BUDGET PROGRAM

Enter your salary for this month: $2000

MENU
1. Add expense
2. Load sample data
3. Display summary
4. Recategorise expense
5. Display all expenses
6. Exit

Choose an option (1-6): 1

ADD EXPENSE
Enter expense amount: $200
What did you spend it on? Dinner

Automatic categorisation:
Description : Dinner
Category    : Food

Expense added successfully.

MENU
1. Add expense
2. Load sample data
3. Display summary
4. Recategorise expense
5. Display all expenses
6. Exit

Choose an option (1-6): 1

ADD EXPENSE
Enter expense amount: $20
What did you spend it on? Book

Automatic categorisation:
Description : Book
Category    : Uncategorised

Expense added successfully.

MENU
1. Add expense
2. Load sample data
3. Display summary
4. Recategorise expense
5. Display all expenses
6. Exit

Choose an option (1-6): 4

ALL EXPENSES
No.  Description                                 Amount            Category
--------------------------------------------------------------

KeyboardInterrupt: Interrupted by user

## Quick reflection

> Double-click to answer.

1. What did your first prompt miss that you had to add?
2. Which single detail made the biggest difference to the output?
3. Could a classmate run your final prompt cold and get a working tool? How do you know?

## Before you submit

- [ ] Your final prompt is one complete block that needs no follow-up questions.
- [ ] The generated code runs top to bottom with no errors.
- [ ] The program makes different decisions for different inputs.
- [ ] You can explain what every part of your prompt is doing and why.
- [ ] You have downloaded the notebook and submitted it as your Week 3 lab ticket.

This is the first brick in a bigger wall. Over the semester your budget logic grows into a finance tracker, and the prompt-writing skill you practise here is one you will lean on the whole way.